In [ ]:
import shutil
import librosa
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [ ]:
def extract_mel(audio_path,
                sr=16000,
                n_mels=128,
                n_fft=1024,
                hop_length=512):

    y, sr = librosa.load(audio_path, sr=sr)

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels
    )

    mel = librosa.power_to_db(mel, ref=np.max)

    return mel.astype(np.float32)

In [ ]:
project_root = Path.cwd().parent

dataset_root = project_root / "data" / "raw" / "pump"
processed_dir = project_root / "data" / "processed"
batch_dir = processed_dir / "batches"

machine_ids = [
    "id_00",
    "id_02",
    "id_04",
    "id_06"
]

batch_size = 256

processed_dir.mkdir(parents=True, exist_ok=True)
batch_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
shutil.rmtree(batch_dir, ignore_errors=True)
batch_dir.mkdir(parents=True, exist_ok=True)

X_batch = []
y_batch = []
batch_idx = 0

def flush_batch():
    global X_batch, y_batch, batch_idx
    if not X_batch:
        return
    np.save(batch_dir / f"X_batch_{batch_idx:04d}.npy", np.array(X_batch))
    np.save(batch_dir / f"y_batch_{batch_idx:04d}.npy", np.array(y_batch))
    X_batch = []
    y_batch = []
    batch_idx += 1

for machine in machine_ids:

    print(f"\nProcessing {machine}")

    normal_dir = dataset_root / machine / "normal"
    abnormal_dir = dataset_root / machine / "abnormal"

    for wav in tqdm(sorted(normal_dir.glob("*.wav"))):
        X_batch.append(extract_mel(wav))
        y_batch.append(0)
        if len(X_batch) >= batch_size:
            flush_batch()

    for wav in tqdm(sorted(abnormal_dir.glob("*.wav"))):
        X_batch.append(extract_mel(wav))
        y_batch.append(1)
        if len(X_batch) >= batch_size:
            flush_batch()

flush_batch()

print(f"\nSaved {batch_idx} batches to {batch_dir}")

In [ ]:
X_parts = sorted(batch_dir.glob("X_batch_*.npy"))
y_parts = sorted(batch_dir.glob("y_batch_*.npy"))

n_total = sum(np.load(f, mmap_mode="r").shape[0] for f in X_parts)
first = np.load(X_parts[0], mmap_mode="r")
n_rows, n_frames = first.shape[1:]
feature_shape = first.shape[1:]

X_path = processed_dir / "X.npy"
y_path = processed_dir / "y.npy"

X_mem = np.lib.format.open_memmap(X_path, mode="w+", dtype=np.float32, shape=(n_total, *feature_shape))
y_mem = np.lib.format.open_memmap(y_path, mode="w+", dtype=np.int64, shape=(n_total,))

start = 0
for xf, yf in zip(X_parts, y_parts):
    x = np.load(xf, mmap_mode="r")
    yy = np.load(yf, mmap_mode="r")
    n = x.shape[0]
    X_mem[start:start + n] = x
    y_mem[start:start + n] = yy
    start += n

X_mem.flush()
y_mem.flush()

print("Feature shape:", X_mem.shape)
print("Labels shape:", y_mem.shape)

In [ ]:
import gc

del x, yy
del first
del X_mem, y_mem
gc.collect()

shutil.rmtree(batch_dir, ignore_errors=True)

print("Dataset saved successfully!")
print(processed_dir / "X.npy", "->", (n_total, *feature_shape))
print(processed_dir / "y.npy", "->", (n_total,))
